# Synthetic runtime verification

This notebook verifies the execution package with invented inputs only. It does not
prepare research data, train research candidates, evaluate final events, or establish
Kaggle permissions. Keep the code checkout or extracted code-only source beside the
wheel. Set TRIAGE_CODE_ROOT to that directory and TRIAGE_ROOT to ignored working storage.
See docs/runtime-foundation.md for setup, returned evidence, and checkpoint retrieval.


In [ ]:
import importlib.metadata as metadata
import os
from pathlib import Path
import platform
import sys

CODE_ROOT = Path(os.environ.get("TRIAGE_CODE_ROOT", Path.cwd())).resolve()
ROOT = Path(os.environ.get("TRIAGE_ROOT", (
    "/kaggle/working/slice2" if Path("/kaggle/working").exists() else str(CODE_ROOT)
))).resolve()
INSTALL = os.environ.get("TRIAGE_INSTALL", "0") == "1"
WHEEL = os.environ.get("TRIAGE_WHEEL")
RUN_PRETRAINED = os.environ.get("TRIAGE_PRETRAINED", "0") == "1"
RUN_PAIR = os.environ.get("TRIAGE_PAIR", "0") == "1"
DEVICE = os.environ.get("TRIAGE_DEVICE", "cpu")
RESUME_DIR = os.environ.get("TRIAGE_RESUME_DIR")

print({"python": platform.python_version(), "platform": platform.system(),
       "code_present": (CODE_ROOT / "pyproject.toml").is_file(),
       "kaggle_quota": "Unknown - user verification required",
       "kaggle_session_limits": "Unknown - user verification required"})
assert sys.version_info[:2] == (3, 12), "Python 3.12 required; return runtime evidence if unavailable."
assert (CODE_ROOT / "pyproject.toml").is_file(), "Set TRIAGE_CODE_ROOT to the extracted source."


## Explicit installation

Leave installation disabled when the package is already installed. For Kaggle, provide
the reviewed wheel, retain the supplied CUDA-enabled PyTorch installation, and require
its base version to match the candidate below. A mismatch leaves Kaggle unverified.
After installation, restart the kernel, set TRIAGE_INSTALL=0, and rerun from the top.


In [ ]:
if INSTALL:
    import subprocess
    assert WHEEL and Path(WHEEL).is_file(), "Supply TRIAGE_WHEEL."
    if platform.system() == "Darwin":
        profile = CODE_ROOT / "requirements/macos-py312.txt"
    else:
        from importlib.metadata import version
        assert version("torch").split("+")[0] == "2.15.0", (
            "CUDA PyTorch release differs from the candidate; return evidence before changing it."
        )
        profile = CODE_ROOT / "requirements/common-py312.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(profile)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", WHEEL], check=True)
    raise RuntimeError("Installation complete. Restart the kernel, disable INSTALL, rerun.")


In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)

from disaster_triage.config import RunConfig
from disaster_triage.runtime import doctor
report = doctor(RunConfig(device=DEVICE), ROOT)
print(report)
assert report["python_supported"] and all(report["versions"].values())
# Validate installed pins; CUDA PyTorch's provider build suffix is recorded separately.
from packaging.requirements import Requirement
for line in (CODE_ROOT / "requirements/common-py312.txt").read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    requirement = Requirement(line)
    if requirement.marker is None or requirement.marker.evaluate():
        assert requirement.specifier.contains(metadata.version(requirement.name)), requirement.name
assert metadata.version("torch").split("+")[0] == "2.15.0"


## Offline checks and optional device checks

CPU always runs first. Set TRIAGE_DEVICE=mps or cuda:0 explicitly for another device.
Each job has separate attempt accounting and outputs. These checks do not establish
research throughput. Ordinary reruns never start research training.


In [ ]:
from disaster_triage.smoke import run_offline, resume_probe, run_pretrained
cpu_result = run_offline(RunConfig(device="cpu"), ROOT, CODE_ROOT)
print(cpu_result)
device_result = None
if DEVICE != "cpu":
    device_result = run_offline(RunConfig(device=DEVICE), ROOT, CODE_ROOT)
    print(device_result)


In [ ]:
if RUN_PRETRAINED:
    print(run_pretrained(RunConfig(device=DEVICE), ROOT, CODE_ROOT))
else:
    print("Pretrained integration is opt-in with TRIAGE_PRETRAINED=1.")


In [ ]:
if RUN_PAIR:
    import torch
    assert torch.cuda.device_count() >= 2, "Two CUDA devices required for paired verification."
    commands = [
        [sys.executable, "-m", "disaster_triage", "smoke", "--root", str(ROOT),
         "--source-root", str(CODE_ROOT), "--device", f"cuda:{index}"]
        for index in (0, 1)
    ]
    workers = [subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
               for command in commands]
    for worker in workers:
        output, error = worker.communicate()
        assert worker.returncode == 0, "Synthetic device worker failed; inspect private output."
        print(output)


## Retrieved checkpoint verification

Retrieve the verification directory shown above, including the checkpoint, reference,
expectation, resolved configuration and source snapshot. End the session. In a fresh
session using identical code/dependencies/device, supply its retrieved path through
TRIAGE_RESUME_DIR. Do not point at an old ephemeral working directory.


In [ ]:
if RESUME_DIR:
    print(resume_probe(RunConfig(device=DEVICE), ROOT, CODE_ROOT, Path(RESUME_DIR)))
else:
    print("Fresh-session checkpoint retrieval/restoration: Unknown - user verification required.")


In [ ]:
from disaster_triage.accounting import budget_summary
from disaster_triage.provenance import save_json
summary = {
    "runtime": report,
    "cpu": cpu_result,
    "device": device_result,
    "accounting": budget_summary(RunConfig().paths(ROOT)["artifact_dir"]),
    "quota_before_after": "Unknown - user verification required",
    "session_limit": "Unknown - user verification required",
    "retrieved_resume_requested": bool(RESUME_DIR),
}
save_json(RunConfig().paths(ROOT)["artifact_dir"] / "verification-summary.json", summary)
print(summary)
